# Geography Chatbot

A short, geography-only chatbot built with LangGraph. In Google Colab, add an OPENAI_API_KEY secret and allow this notebook to access it before running the setup cells.


In [ ]:
%pip -q install langgraph langchain langchain-openai


In [ ]:
import os
from typing import Annotated, TypedDict

from google.colab import userdata
from langchain.chat_models import init_chat_model
from langchain_core.messages import AnyMessage, HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

try:
    api_key = userdata.get("OPENAI_API_KEY")
except Exception as exc:
    raise RuntimeError(
        "Add OPENAI_API_KEY to Colab Secrets and enable notebook access to it."
    ) from exc

if not api_key:
    raise ValueError("The Colab secret OPENAI_API_KEY is empty.")

os.environ["OPENAI_API_KEY"] = api_key


In [ ]:
model = init_chat_model("openai:gpt-6-luna")

SYSTEM = SystemMessage(
    content=(
        "You are a helpful geography chatbot for students. Answer questions about "
        "geography only. If a request is outside geography, politely explain that "
        "you can only help with geography."
    )
)

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

def chatbot(state: State):
    reply = model.invoke([SYSTEM, *state["messages"]])
    return {"messages": [reply]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile()


## Chat
Enter stop to end the conversation.


In [ ]:
history = []

while True:
    text = input("You: ")
    if text.strip().lower() == "stop":
        print("Chat ended.")
        break

    result = graph.invoke({"messages": history + [HumanMessage(content=text)]})
    history = result["messages"]
    print("Assistant:", history[-1].content)
